In [1]:
from code_utils import *

In [ ]:
def computing_coefficients_martingale_alfonsi_rnn(N, N_bar, nneurons, discounted_asset, discounted_payoff, paths, asset, sigma, seed):

    h = nneurons + 1
    coefficients = np.zeros((N, N_bar, h))    
    martingale_increments = np.zeros((N, paths))
    weights_store = [None] * N                 # weights per block, reused in pricing
    stats_store = [None] * N                   # stats per block, reused in pricing
    np.random.seed(seed)         
    # One draw
    A = sigma * np.random.standard_normal(nneurons)
    b = sigma * np.random.standard_normal(nneurons)
    shared_w = (A, b)
        
    for i in range(N - 1, -1, -1):
        Z = discounted_payoff[(i + 1):, :]
        if i + 1 < N:
            sum_to_remove = np.vstack([np.zeros((1, paths)), np.cumsum(martingale_increments[(i + 1):, :], axis=0)])
        else:
            sum_to_remove = np.zeros((Z.shape[0], paths))

        vector_to_maximize = np.max(Z - sum_to_remove, axis=0)
        Z_i = discounted_payoff[i, :]
        target = vector_to_maximize - Z_i                                   # (paths,), COMMON to all j

        asset_block = asset[i * N_bar:(i + 1) * N_bar, :]
        dblk = discounted_asset[i * N_bar:(i + 1) * N_bar + 1, :]
        difference_discounted_asset = dblk[1:, :] - dblk[:-1, :]            # (N_bar, paths)
        # Standardized asset block for better numerical stability
        stats_store[i] = fit_log_standardizer(asset_block)
        asset_block_std = apply_log_standardizer(asset_block, stats_store[i])	
        
        rnn_basis, w = random_basis(asset_block_std, nneurons, sigma, seed, shared_w)         # (h, N_bar, paths)
        weights_store[i] = w
        pre_X = rnn_basis * difference_discounted_asset[np.newaxis, :, :]   # (h, N_bar, paths)

        incr_i = np.zeros(paths)
        for j in range(N_bar):                                             	# the N_bar decoupled systems
            dXj = pre_X[:, j, :]                                            # (h, paths) = Delta X_{i,j}
            alpha_j = np.linalg.lstsq(dXj.T, target, rcond=None)[0]         # (h,)
            coefficients[i, j, :] = alpha_j
            incr_i += alpha_j @ dXj
        martingale_increments[i, :] = incr_i

    martingale = np.cumsum(martingale_increments, axis=0)
    return coefficients, martingale, weights_store, stats_store

# Since in this function weights are always provided, the seed is not used. However, it is kept for consistency with the other functions.
def alfonsi_martingale_rnn(N, N_bar, nneurons, discounted_asset, coefficients, paths, asset, sigma, weights_store, stats_store, seed):
    martingale_increments = np.zeros((N, paths))
    for i in range(N - 1, -1, -1):
        asset_block = asset[i * N_bar:(i + 1) * N_bar, :]
        dblk = discounted_asset[i * N_bar:(i + 1) * N_bar + 1, :]
        difference_discounted_asset = dblk[1:, :] - dblk[:-1, :]

        # Standardized asset block for better numerical stability
        asset_block_std = apply_log_standardizer(asset_block, stats_store[i])	

        rnn_basis, _ = random_basis(asset_block_std, nneurons, sigma, seed, weights=weights_store[i])      # SAME weights as training
        pre_X = rnn_basis * difference_discounted_asset[np.newaxis, :, :]
        incr_i = np.zeros(paths)
        for j in range(N_bar):
            dXj = pre_X[:, j, :]
            incr_i += coefficients[i, j, :] @ dXj
        martingale_increments[i, :] = incr_i
    return np.cumsum(martingale_increments, axis=0)




# Trying to understand and implement randomized neural network!
# 1) Important to understand if in this case we have to consider ITM paths or all paths -> All paths
# 2) For each t_ij we call this again ?
# 3) Do we change the random pair (A,b) for each t_ij ? No, it's like the initial basis, it stays the same for all t_ij
# 4) Important to remember how we save the coefficients!

 # The pair of weights (W,b) is independent from the path and subintervals! - > In the present case we are not doing this

In [3]:
# Parameters
N = 50    		# Number of exercising dates
N_bar = 5       # Number of intervals between two exercising dates
K = 100   		# Strike price
r = 0.05  		# Risk-free rate
sigma = 0.3     # Volatility
T = 0.2   		# Maturity

seed = 42    	# Seed for reproducibility
seed_weights = 12345

paths = 10**5 	# Number of Monte Carlo paths
pricing_paths = 10**5
nneurons = 9 	# Number of neurons in the randomized neural network ( -1 )
sigma_basis = sigma  # Standard deviation for the random weights in the basis functions

real_prices = np.array([
    40.00000000, 39.00000000, 38.00000000, 37.00000000, 36.00000000,
    35.00000000, 34.00000000, 33.00000000, 32.00000000, 31.00000000,
    30.00000000, 29.00000000, 28.00000000, 27.00000000, 26.00000000,
    25.00000000, 24.00000000, 23.00000000, 22.00000000, 21.00000000,
    20.00070685, 19.01347004, 18.04313091, 17.09109638, 16.15883366,
    15.24783858, 14.35960625, 13.49560169, 12.65723381, 11.84582143,
    11.06257234, 10.30856320,  9.58471980,  8.89180771,  8.23041673,
     7.60094690,  7.00361025,  6.43844700,  5.90532237,  5.40390521,
     4.93370425,  4.49410547,  4.08432251,  3.70344748,  3.35050142,
     3.02435964,  2.72386986,  2.44779995,  2.19486933,  1.96380311,
     1.75326609,  1.56197956,  1.38862261,  1.23195119,  1.09070376,
     0.96370545,  0.84978944,  0.74786820,  0.65689777,  0.57588855,
     0.50392803,  0.44013960,  0.38373063,  0.33395543,  0.29012326,
     0.25161274,  0.21784490,  0.18829301,  0.16248633,  0.13999277,
     0.12042199,  0.10342780,  0.08869869,  0.07595400,  0.06494518,
     0.05545341,  0.04728321,  0.04026147,  0.03423616,  0.02907429,
     0.02465918
])

prices = np.empty(81)
uncertainty = np.empty(81)
percentage_errors = np.empty(81)
coefficients = {}

fixed_map = True  # Use the same random basis for all time steps
for S0 in range(60, 141):

    # Temporal grid
    exercise_time = np.linspace(0, T, N + 1)
    exercise_indices = range(0, N_bar * N + 1, N_bar)
    dense_grid = np.linspace(0, T, N_bar * N + 1)
    
    discount_factor = np.exp(-r * dense_grid)
    discount_factor_exercise_time = np.exp(-r * exercise_time)

    # Training paths 
    asset = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths, seed)
    discounted_asset = asset * discount_factor[:, np.newaxis]
    discounted_payoff = (np.maximum(K - asset[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])

    # Pricing paths 
    asset_pricing = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], pricing_paths, seed + 1)
    discounted_asset_pricing = asset_pricing * discount_factor[:, np.newaxis]
    discounted_payoff_pricing = (np.maximum(K - asset_pricing[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])

    prices[S0 - 60], _, uncertainty[S0 - 60], coefficients[S0],_ = upper_bound_alfonsi_rnn( asset, asset_pricing, discounted_asset, discounted_asset_pricing, discounted_payoff, discounted_payoff_pricing, paths, pricing_paths, N, N_bar, nneurons, sigma_basis, seed_weights)
    percentage_errors[S0- 60] = np.abs(prices[S0-60] - real_prices[S0 - 60]) / real_prices[S0 - 60] * 100
    print(f"S0= {S0}, Estimated Price: {prices[S0-60]:.5f}, Real Price: {real_prices[S0-60]:.5f}, Percentage Error: {percentage_errors[S0-60]:.2f}%, Uncertainty: {uncertainty[S0-60]:.5f}")

S0= 60, Estimated Price: 40.00721, Real Price: 40.00000, Percentage Error: 0.02%, Uncertainty: 0.00034
S0= 61, Estimated Price: 39.00747, Real Price: 39.00000, Percentage Error: 0.02%, Uncertainty: 0.00035
S0= 62, Estimated Price: 38.00775, Real Price: 38.00000, Percentage Error: 0.02%, Uncertainty: 0.00035
S0= 63, Estimated Price: 37.00807, Real Price: 37.00000, Percentage Error: 0.02%, Uncertainty: 0.00036
S0= 64, Estimated Price: 36.00842, Real Price: 36.00000, Percentage Error: 0.02%, Uncertainty: 0.00038
S0= 65, Estimated Price: 35.00881, Real Price: 35.00000, Percentage Error: 0.03%, Uncertainty: 0.00039
S0= 66, Estimated Price: 34.00920, Real Price: 34.00000, Percentage Error: 0.03%, Uncertainty: 0.00041
S0= 67, Estimated Price: 33.00959, Real Price: 33.00000, Percentage Error: 0.03%, Uncertainty: 0.00041
S0= 68, Estimated Price: 32.01014, Real Price: 32.00000, Percentage Error: 0.03%, Uncertainty: 0.00041
S0= 69, Estimated Price: 31.01096, Real Price: 31.00000, Percentage Error

In [4]:
np.savez('rnn_alfonsi_upper_results.npz', S0=np.arange(60,141), price=prices, error=percentage_errors, uncertainty=uncertainty)

In [5]:
# # Computing martingale for control variate/upper bound using Alfonsi method knowing the coefficients
# def alfonsi_martingale_rnn(N, N_bar, nneurons, discounted_asset, coefficients, paths, asset, sigma, weights_store):
#     martingale_increments = np.zeros((N, paths))
#     L  = (nneurons + 1) * (N_bar)
#     for i in range(N-1, -1, -1):
#         delta_X_all = np.zeros((L, paths)) 

#         asset_block = asset[i * N_bar : (i + 1) * N_bar, :] # by default (i+1)*N_bar is not included
#         discounted_asset_block = discounted_asset[i * N_bar:(i + 1) * N_bar + 1, :] # shape (N_bar + 1, paths)
#         difference_discounted_asset = discounted_asset_block[1:, :] - discounted_asset_block[:-1, :] # shape (N_bar, paths)

#         rnn_basis, _ = random_basis(asset_block, nneurons, sigma, weights=weights_store[i])   # REUSE, don't redraw
#         pre_X = rnn_basis * difference_discounted_asset[np.newaxis, :, :] # shape (nneurons + 1, N_bar, paths)
#         delta_X_all = pre_X.reshape(L, paths) # shape ((nneurons + 1) * N_bar, paths)

#         martingale_increments[i, :] = coefficients[i,:] @ delta_X_all                
#     martingale = np.cumsum(martingale_increments, axis=0)
#     return martingale



# # Training phase Alfonsi
# def computing_coefficients_martingale_alfonsi_rnn(N, N_bar, nneurons, discounted_asset, discounted_payoff, paths, asset, sigma):

#     L = (nneurons + 1) * (N_bar)
#     coefficients = np.zeros((N, L))
#     martingale_increments = np.zeros((N, paths))
#     martingale = np.zeros((N, paths))
    
#     weights_store = [None] * N                      


#     for i in range(N-1, -1, -1):

#         Z = discounted_payoff[(i+1):, :] 
#         if i + 1 < N:
#             sum_to_remove = np.vstack([np.zeros((1, paths)), np.cumsum(martingale_increments[(i+1):, :], axis=0)])
#         else:
#             sum_to_remove = np.zeros((Z.shape[0], paths))
            
#         vector_to_maximize = np.max(Z - sum_to_remove, axis=0)

#         # Z_i: discounted payoff at current exercise time i
#         Z_i = discounted_payoff[i, :]

#         asset_block = asset[i * N_bar : (i + 1) * N_bar, :] # by default (i+1)*N_bar is not included
#         discounted_asset_block = discounted_asset[i * N_bar:(i + 1) * N_bar + 1, :] # shape (N_bar + 1, paths)
#         difference_discounted_asset = discounted_asset_block[1:, :] - discounted_asset_block[:-1, :] # shape (N_bar, paths)

#         rnn_basis, w = random_basis(asset_block, nneurons, sigma)
#         weights_store[i] = w
        
#         #rnn_basis = random_basis(asset_block, nneurons, sigma) # shape (nneurons + 1, N_bar, paths)
#         pre_X = rnn_basis * difference_discounted_asset[np.newaxis, :, :] # shape (nneurons + 1, N_bar, paths)
#         delta_X_all = pre_X.reshape(L, paths) # shape ((nneurons + 1) * N_bar, paths)

#         # Using martingale property it is possible to obtain
#         # We want know to define the linear system we aim to solve
#         b = np.mean(delta_X_all * (vector_to_maximize - Z_i)[np.newaxis, :], axis=1) # shape (L,)
#         A = delta_X_all @ delta_X_all.T / paths # shape (L,L)
        
#         coefficients[i, :] = np.linalg.lstsq(A, b, rcond=None)[0] # shape (L,)
    
#         martingale_increments[i, :] = coefficients[i,:] @ delta_X_all # where delta_X_all has shape(L, paths)

#     martingale = np.cumsum(martingale_increments, axis=0) 
#     return coefficients, martingale, weights_store    

# # Defining the activation function 
# def activation(x):
#     y = x / (1 + np.exp(-x))
#     return y

# # With random basis we mean (1,csi_1, ..., csi_nneurons) where csi_i = activation(A_i * S + b_i) and A_i, b_i are random weights and bias
# # S is going to represent the asset_block of shape (N_bar, paths)
# def random_basis(S, nneurons, sigma, weights=None):
#     if weights is None:
#         A = sigma * np.random.standard_normal(nneurons) # shape (nneurons,)
#         b = sigma * np.random.standard_normal(nneurons) # shape (nneurons,)
#     else:
#         A, b = weights
#     activ = activation(A[:, np.newaxis, np.newaxis] * S[np.newaxis, :, :] + b[:, np.newaxis, np.newaxis])
#     const = np.ones((1, S.shape[0], S.shape[1]))
#     basis = np.concatenate([const, activ], axis=0)
#     return basis, (A, b)          



# def upper_bound_alfonsi_indicator_mod_rnn(asset, asset_pricing, discounted_asset, discounted_asset_pricing, discounted_payoff, discounted_payoff_pricing, paths, pricing_paths, N, N_bar, nneurons, sigma):
#     #Training
#     coefficients, martingale_train, weights_store = computing_coefficients_martingale_alfonsi_rnn(N, N_bar, nneurons, discounted_asset, discounted_payoff, paths, asset, sigma)
#     max_path_train = np.max(discounted_payoff - np.vstack([np.zeros((1, paths)), martingale_train]), axis=0)
#     option_price_upper = np.mean(max_path_train)
#     # Pricing
#     martingale_pricing = alfonsi_martingale_rnn(N, N_bar, nneurons, discounted_asset_pricing, coefficients, pricing_paths,asset_pricing, sigma, weights_store)
#     max_per_path = np.max(discounted_payoff_pricing - np.vstack([np.zeros((1, pricing_paths)), martingale_pricing]), axis=0)
#     option_price_upper_hat = np.mean(max_per_path)
#     uncertainty = np.std(max_per_path, ddof=1) / np.sqrt(pricing_paths)

#     return option_price_upper_hat, option_price_upper, uncertainty, coefficients 



In [7]:
# Parameters
N = 50    		# Number of exercising dates
N_bar = 5       # Number of intervals between two exercising dates
K = 100   		# Strike price
r = 0.05  		# Risk-free rate
sigma = 0.3     # Volatility
T = 0.2   		# Maturity

seed = 42    	# Seed for reproducibility
seed_weights = 12345

paths = 10**5 	# Number of Monte Carlo paths
pricing_paths = 10**5
nneurons = 9 	# Number of neurons in the randomized neural network ( -1 )
sigma_basis = sigma  # Standard deviation for the random weights in the basis functions

real_prices = np.array([
    40.00000000, 39.00000000, 38.00000000, 37.00000000, 36.00000000,
    35.00000000, 34.00000000, 33.00000000, 32.00000000, 31.00000000,
    30.00000000, 29.00000000, 28.00000000, 27.00000000, 26.00000000,
    25.00000000, 24.00000000, 23.00000000, 22.00000000, 21.00000000,
    20.00070685, 19.01347004, 18.04313091, 17.09109638, 16.15883366,
    15.24783858, 14.35960625, 13.49560169, 12.65723381, 11.84582143,
    11.06257234, 10.30856320,  9.58471980,  8.89180771,  8.23041673,
     7.60094690,  7.00361025,  6.43844700,  5.90532237,  5.40390521,
     4.93370425,  4.49410547,  4.08432251,  3.70344748,  3.35050142,
     3.02435964,  2.72386986,  2.44779995,  2.19486933,  1.96380311,
     1.75326609,  1.56197956,  1.38862261,  1.23195119,  1.09070376,
     0.96370545,  0.84978944,  0.74786820,  0.65689777,  0.57588855,
     0.50392803,  0.44013960,  0.38373063,  0.33395543,  0.29012326,
     0.25161274,  0.21784490,  0.18829301,  0.16248633,  0.13999277,
     0.12042199,  0.10342780,  0.08869869,  0.07595400,  0.06494518,
     0.05545341,  0.04728321,  0.04026147,  0.03423616,  0.02907429,
     0.02465918
])


for S0 in range(60, 141):
    price_estimate, uncertainty_estimate = rnn_upper_bound_american_put_alfonsi(K, S0, r, sigma, sigma_basis, T, N, N_bar, nneurons, paths, pricing_paths, seed, seed_weights)
    real_price = real_prices[S0 - 60]
    percentage_error = np.abs(price_estimate - real_price) / real_price * 100
    print(f"S0= {S0}, Estimated Price: {price_estimate:.5f}, Real Price: {real_price:.5f}, Percentage Error: {percentage_error:.2f}%, Uncertainty: {uncertainty_estimate:.5f}")
	

S0= 60, Estimated Price: 40.00721, Real Price: 40.00000, Percentage Error: 0.02%, Uncertainty: 0.00034
S0= 61, Estimated Price: 39.00747, Real Price: 39.00000, Percentage Error: 0.02%, Uncertainty: 0.00035
S0= 62, Estimated Price: 38.00775, Real Price: 38.00000, Percentage Error: 0.02%, Uncertainty: 0.00035
S0= 63, Estimated Price: 37.00807, Real Price: 37.00000, Percentage Error: 0.02%, Uncertainty: 0.00036
S0= 64, Estimated Price: 36.00842, Real Price: 36.00000, Percentage Error: 0.02%, Uncertainty: 0.00038
S0= 65, Estimated Price: 35.00881, Real Price: 35.00000, Percentage Error: 0.03%, Uncertainty: 0.00039
S0= 66, Estimated Price: 34.00920, Real Price: 34.00000, Percentage Error: 0.03%, Uncertainty: 0.00041
S0= 67, Estimated Price: 33.00959, Real Price: 33.00000, Percentage Error: 0.03%, Uncertainty: 0.00041
S0= 68, Estimated Price: 32.01014, Real Price: 32.00000, Percentage Error: 0.03%, Uncertainty: 0.00041
S0= 69, Estimated Price: 31.01096, Real Price: 31.00000, Percentage Error